# ELIMINAR ESTACIONES CON POCAS MEDICIONES

## DESPUES -> AÑADIR DATOS DE TRAFICO

In [2]:
import pandas as pd

df = pd.read_csv("datos/cont_meteo_juntos_2021_2024.csv")

df

,ESTACION,DISTRITO,FECHA_HORA,NO,NO2,NOx,BEN,CO,EBE,O3,...,PM2_5,SO2,TOL,TEMP,HUM_REL,PRECIPITACION,PRES_BARIOMETRICA,RAD_SOL,DIR_VIENT,VEL_VIENT
0,4,9,2021-01-01 01:00:00,1.0,10.0,12.0,NaN,0.2,NaN,NaN,...,NaN,6.0,NaN,3.5,NaN,NaN,NaN,NaN,NaN,NaN
1,4,9,2021-01-01 02:00:00,1.0,16.0,18.0,NaN,0.2,NaN,NaN,...,NaN,6.0,NaN,3.2,NaN,NaN,NaN,NaN,NaN,NaN
2,4,9,2021-01-01 03:00:00,1.0,7.0,9.0,NaN,0.2,NaN,NaN,...,NaN,5.0,NaN,3.8,NaN,NaN,NaN,NaN,NaN,NaN
3,4,9,2021-01-01 04:00:00,1.0,5.0,6.0,NaN,0.2,NaN,NaN,...,NaN,5.0,NaN,3.7,NaN,NaN,NaN,NaN,NaN,NaN
4,4,9,2021-01-01 05:00:00,1.0,6.0,8.0,NaN,0.2,NaN,NaN,...,NaN,5.0,NaN,3.5,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
615436,60,8,2023-12-31 20:00:00,5.0,30.0,37.0,NaN,NaN,NaN,37.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
615437,60,8,2023-12-31 21:00:00,6.0,35.0,44.0,NaN,NaN,NaN,36.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
615438,60,8,2023-12-31 22:00:00,4.0,37.0,44.0,NaN,NaN,NaN,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
615439,60,8,2023-12-31 23:00:00,2.0,20.0,23.0,NaN,NaN,NaN,51.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# COMBINACIONES DE ESTACION Y DISTRITO
combinaciones = df.groupby(['ESTACION', 'DISTRITO']).size().reset_index(name='Count')

print(combinaciones)

    ESTACION  DISTRITO  Count
0          4         9  21727
1          8         3  26070
2         11         5  24825
3         16        15  23662
4         17        17  26013
5         18        11  26122
6         24         9  25744
7         27        21  25979
8         35         1  24427
9         36        14  25844
10        38         6  26206
11        39         8  25941
12        40        18  26125
13        47         2  25915
14        48         4  26196
15        49         3  25851
16        50         5  26232
17        54        18  25968
18        55        21  25938
19        56        12  26136
20        57        16  26213
21        58         8  26055
22        59        21  26035
23        60         8  26217


In [4]:
import pandas as pd

columnas = ['NO', 'NO2', 'NOx', 'BEN', 'CO', 'EBE', 'O3', 'PM10', 'PM2_5', 'SO2', 'TOL']  # Puedes agregar más columnas aquí si es necesario

valores_nan_por_estacion = df.groupby('ESTACION')[columnas].apply(lambda x: x.isna().sum())

total_por_estacion = df['ESTACION'].value_counts().reset_index()
total_por_estacion.columns = ['ESTACION', 'TOTAL_MUESTRAS']

valores_nan_por_estacion = pd.merge(valores_nan_por_estacion, total_por_estacion, on='ESTACION')

for columna in columnas:
    valores_nan_por_estacion[columna] = (valores_nan_por_estacion[columna] / valores_nan_por_estacion['TOTAL_MUESTRAS']) * 100

valores_nan_por_estacion = valores_nan_por_estacion[['ESTACION'] + columnas]

valores_nan_por_estacion = valores_nan_por_estacion.sort_values(by='ESTACION').reset_index(drop=True)

valores_nan_por_estacion = valores_nan_por_estacion.round(2)

valores_nan_por_estacion

,ESTACION,NO,NO2,NOx,BEN,CO,EBE,O3,PM10,PM2_5,SO2,TOL
0,4,8.15,7.71,9.00,100.00,7.14,100.00,100.00,100.00,100.00,7.22,100.00
1,8,5.38,5.30,5.22,8.01,4.81,7.91,5.95,11.71,11.49,6.12,7.31
2,11,3.83,4.05,4.49,6.88,100.00,6.44,100.00,100.00,100.00,100.00,8.69
3,16,8.76,3.46,2.69,100.00,100.00,100.00,5.86,100.00,100.00,100.00,100.00
4,17,4.09,3.85,5.79,100.00,100.00,100.00,9.47,100.00,100.00,100.00,100.00
5,18,4.59,5.50,5.23,9.48,71.02,8.86,5.14,10.67,100.00,88.25,9.50
6,24,5.86,6.50,7.74,7.71,100.00,5.44,9.09,6.65,4.76,100.00,10.01
7,27,4.02,4.38,4.09,100.00,100.00,100.00,4.28,100.00,100.00,100.00,100.00
8,35,9.71,9.73,6.99,100.00,7.85,100.00,7.23,100.00,100.00,7.39,100.00
9,36,5.00,3.49,4.46,100.00,100.00,100.00,100.00,4.62,100.00,12.03,100.00


## Eliminar estaciones donde PM10 = 100

In [5]:
filas_a_eliminar = valores_nan_por_estacion[valores_nan_por_estacion['PM10'] == 100.00]
print("Valores de la columna 'estacion' de las filas a eliminar:")
print(filas_a_eliminar['ESTACION'])

Valores de la columna 'estacion' de las filas a eliminar:
0      4
2     11
3     16
4     17
7     27
8     35
11    39
15    49
17    54
21    58
22    59
Name: ESTACION, dtype: int64


### ESTACIONES QUE  HAY QUE ELIMINAR

4

11

16

17

27

35

39

49

54

58

59

In [6]:
valores_nan_por_estacion.drop(valores_nan_por_estacion[valores_nan_por_estacion['PM10'] == 100.00].index, inplace=True)

valores_nan_por_estacion

,ESTACION,NO,NO2,NOx,BEN,CO,EBE,O3,PM10,PM2_5,SO2,TOL
1,8,5.38,5.30,5.22,8.01,4.81,7.91,5.95,11.71,11.49,6.12,7.31
5,18,4.59,5.50,5.23,9.48,71.02,8.86,5.14,10.67,100.00,88.25,9.50
6,24,5.86,6.50,7.74,7.71,100.00,5.44,9.09,6.65,4.76,100.00,10.01
9,36,5.00,3.49,4.46,100.00,100.00,100.00,100.00,4.62,100.00,12.03,100.00
10,38,4.72,6.28,4.46,5.85,100.00,7.15,100.00,10.66,10.78,100.00,6.18
12,40,3.30,3.35,5.12,100.00,100.00,100.00,100.00,5.63,100.00,100.00,100.00
13,47,3.98,4.26,2.96,100.00,100.00,100.00,100.00,6.13,6.88,100.00,100.00
14,48,2.83,2.10,2.95,100.00,100.00,100.00,100.00,6.66,5.72,100.00,100.00
16,50,3.80,4.38,3.71,100.00,100.00,100.00,100.00,6.24,5.99,100.00,100.00
18,55,4.52,4.11,4.95,13.00,100.00,9.65,100.00,8.17,100.00,100.00,11.06


### MEDICIONES A ELIMINAR

BEN

CO

EBE

O3

SO2

TOL

# ELIMINACIÓN DE COLUMNAS DE CONTAMINACIÓN Y FILAS CON ESTACIONES

In [7]:
df = pd.read_csv("datos/cont_meteo_juntos_2021_2024.csv")

df.shape

(615441, 21)

In [8]:
df = df.drop('BEN', axis=1)
df = df.drop('CO', axis=1)
df = df.drop('EBE', axis=1)
df = df.drop('O3', axis=1)
df = df.drop('SO2', axis=1)
df = df.drop('TOL', axis=1)

df.shape

(615441, 15)

In [9]:
df = df.drop(df[df['ESTACION'] == 4].index)
df = df.drop(df[df['ESTACION'] == 11].index)
df = df.drop(df[df['ESTACION'] == 16].index)
df = df.drop(df[df['ESTACION'] == 17].index)
df = df.drop(df[df['ESTACION'] == 27].index)
df = df.drop(df[df['ESTACION'] == 35].index)
df = df.drop(df[df['ESTACION'] == 39].index)
df = df.drop(df[df['ESTACION'] == 49].index)
df = df.drop(df[df['ESTACION'] == 54].index)
df = df.drop(df[df['ESTACION'] == 58].index)
df = df.drop(df[df['ESTACION'] == 59].index)
df.shape

(338958, 15)

Casi la mitad de filas

In [10]:
# Verificar que es el mismo df que antes

columnas = ['NO', 'NO2', 'NOx', 'PM10', 'PM2_5'] 

valores_nan_por_estacion = df.groupby('ESTACION')[columnas].apply(lambda x: x.isna().sum())

total_por_estacion = df['ESTACION'].value_counts().reset_index()
total_por_estacion.columns = ['ESTACION', 'TOTAL_MUESTRAS']

valores_nan_por_estacion = pd.merge(valores_nan_por_estacion, total_por_estacion, on='ESTACION')

for columna in columnas:
    valores_nan_por_estacion[columna] = (valores_nan_por_estacion[columna] / valores_nan_por_estacion['TOTAL_MUESTRAS']) * 100

valores_nan_por_estacion = valores_nan_por_estacion[['ESTACION'] + columnas]

valores_nan_por_estacion = valores_nan_por_estacion.sort_values(by='ESTACION').reset_index(drop=True)

valores_nan_por_estacion = valores_nan_por_estacion.round(2)

valores_nan_por_estacion

#CORRECTO

,ESTACION,NO,NO2,NOx,PM10,PM2_5
0,8,5.38,5.30,5.22,11.71,11.49
1,18,4.59,5.50,5.23,10.67,100.00
2,24,5.86,6.50,7.74,6.65,4.76
3,36,5.00,3.49,4.46,4.62,100.00
4,38,4.72,6.28,4.46,10.66,10.78
5,40,3.30,3.35,5.12,5.63,100.00
6,47,3.98,4.26,2.96,6.13,6.88
7,48,2.83,2.10,2.95,6.66,5.72
8,50,3.80,4.38,3.71,6.24,5.99
9,55,4.52,4.11,4.95,8.17,100.00


In [11]:
df.to_csv("datos/CONT_METEO_REDUCIDO_2021_2024.CSV")

# LA MITAD DE MEMORIA EN DISCO